# Geometry-V3 keyed Q/K active writer P0D2
One controlled SD3.5 final inner-Q narrowing run. Outputs are operational evidence with `science_denominator=0`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import datetime as dt
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
from google.colab import userdata

REPOSITORY = 'https://github.com/RICHAAARC/CEG-WM.git'
P0D2_RUNNER_EXACT = 'a27a940ae1ef4d1141925c6304caab55b89ec999'
RUNNER_RELATIVE_PATH = 'experiments/run_geometry_v3_qk_active_writer_p0d2.py'
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V3/P0D2')
MAX_CONTROL_BYTES = 1024

hf_token = userdata.get('HF_TOKEN')
geometry_key = userdata.get('CEGWM_GEOMETRY_KEY')
if not isinstance(hf_token, str) or not hf_token.strip():
    raise RuntimeError('HF_TOKEN userdata is required')
if not isinstance(geometry_key, str) or not geometry_key.strip():
    raise RuntimeError('CEGWM_GEOMETRY_KEY userdata is required')

checkout = Path(tempfile.mkdtemp(prefix='cegwm-geometry-v3-p0d2-checkout-'))
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(checkout)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['git', 'checkout', '--detach', P0D2_RUNNER_EXACT], cwd=checkout, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
resolved = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=checkout, check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=checkout, check=True, capture_output=True, text=True).stdout
if resolved != P0D2_RUNNER_EXACT or dirty:
    raise RuntimeError('P0D2 detached checkout identity differs')
runner_path = checkout / RUNNER_RELATIVE_PATH
if not runner_path.is_file():
    raise RuntimeError('P0D2 runner is absent from the bound checkout')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

timestamp = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
drive_directory = DRIVE_ROOT / f'Geometry-V3-P0D2-{P0D2_RUNNER_EXACT[:12]}-{timestamp}'
if drive_directory.exists():
    raise FileExistsError('P0D2 Drive output already exists')
plan = {
    'expected_exact': P0D2_RUNNER_EXACT,
    'execution_exact': P0D2_RUNNER_EXACT,
    'output_directory': str(drive_directory),
}
plan_file = Path(tempfile.mkdtemp(prefix='cegwm-geometry-v3-p0d2-plan-')) / 'plan.json'
plan_file.write_text(json.dumps(plan, sort_keys=True, separators=(',', ':')), encoding='utf-8')
child_env = os.environ.copy()
child_env['HF_TOKEN'] = hf_token
child_env['CEGWM_GEOMETRY_KEY'] = geometry_key
hf_token = ''
geometry_key = ''
read_fd, write_fd = os.pipe()
try:
    child = subprocess.Popen(
        [sys.executable, str(runner_path), '--plan', str(plan_file), '--control-fd', str(write_fd)],
        cwd=checkout, env=child_env, pass_fds=(write_fd,),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    os.close(write_fd)
    write_fd = -1
    runner_rc = child.wait(timeout=7200)
    control_bytes = os.read(read_fd, MAX_CONTROL_BYTES + 1)
finally:
    child_env.pop('HF_TOKEN', None)
    child_env.pop('CEGWM_GEOMETRY_KEY', None)
    if write_fd >= 0:
        os.close(write_fd)
    os.close(read_fd)
if len(control_bytes) > MAX_CONTROL_BYTES:
    raise RuntimeError('P0D2 control receipt exceeds bound')
control = json.loads(control_bytes) if control_bytes else {}
artifact_status = 'complete' if control.get('status') == 'success' else 'unavailable'
terminal = {
    'runner_execution_identity': {'commit': P0D2_RUNNER_EXACT, 'clean': True},
    'runner_rc': runner_rc,
    'drive_directory': str(drive_directory),
    'handoff_status': control.get('status'),
    'status': control.get('p0d2_status'),
    'run_id': control.get('run_id'),
    'artifact_status': artifact_status,
    'failure_point': control.get('failure_point'),
    'error_class': control.get('error_class'),
    'counters': control.get('counters'),
    'science_denominator': 0,
}
print('CEGWM_GEOMETRY_V3_QK_P0D2_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
if runner_rc != 0 or control.get('status') != 'success':
    raise RuntimeError('Geometry-V3 P0D2 runner failed; inspect bounded terminal fields')